# Entrega 2 - Transformacao, Feature Engineering e Divisao Train/Test

Este notebook implementa a versao revisada da Entrega 2, considerando o pedido da professora: divisao em treino/teste antes das transformacoes que aprendem paremetros, prevencao de vazamento de dados, feature engineering, normalizacao e geracao de uma amostra do dataset.

A correcao principal desta versao é refazer a imputacao KNN da faixa etaria depois da divisao treino/teste. O arquivo `drug_events_cleaned.csv` ainda contem idade e unidade originais; por isso a faixa etaria é recalculada antes da imputacao, e o KNN é treinado somente com dados do conjunto de treinamento.


## 1. Carregamento dos Dados Base

O CSV limpo é usado como base tabular ja normalizada/agregada a partir do JSON original. Transformacoes que aprendem parametros estatisticos ou padroes dos dados sao refeitas depois da divisao treino/teste.


In [10]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 120)

current_dir = Path.cwd()
project_dir = current_dir.parent if current_dir.name == "notebooks" else current_dir
output_dir = project_dir / "outputs"
input_path = output_dir / "drug_events_cleaned.csv"

if not input_path.exists():
    raise FileNotFoundError(f"Arquivo neo encontrado: {input_path}")

base_df = pd.read_csv(input_path)

expected_columns = [
    "safetyreportid",
    "serious",
    "occurcountry",
    "patient.drug.activesubstance.activesubstancename",
    "patient.drug.drugcharacterization",
    "patient.drug.medicinalproduct",
    "patient.drug.count",
    "patient.patientsex",
    "patient.patientonsetage",
    "patient.patientonsetageunit",
]
missing_columns = [col for col in expected_columns if col not in base_df.columns]
if missing_columns:
    raise ValueError(f"Colunas esperadas ausentes: {missing_columns}")

text_columns = [
    "patient.drug.activesubstance.activesubstancename",
    "patient.drug.drugcharacterization",
    "patient.drug.medicinalproduct",
]
base_df[text_columns] = base_df[text_columns].fillna("unknown").astype(str)
base_df["occurcountry"] = base_df["occurcountry"].fillna("unknown").astype(str)
base_df["patient.patientsex"] = base_df["patient.patientsex"].fillna("unknown").astype(str)
base_df["patient.drug.count"] = pd.to_numeric(base_df["patient.drug.count"], errors="coerce").fillna(0)
base_df["serious"] = base_df["serious"].astype(int)

print(f"Dataset base carregado: {base_df.shape}")
print(f"Arquivo de origem: {input_path}")
base_df.head()


Dataset base carregado: (32309, 12)
Arquivo de origem: c:\Users\Joao Vitor\Desktop\mineracao-dados\datamining-project-unifei\outputs\drug_events_cleaned.csv


,safetyreportid,serious,occurcountry,patient.drug.activesubstance.activesubstancename,patient.drug.drugcharacterization,patient.drug.medicinalproduct,patient.drug.count,patient.patientsex,patient.patientonsetage,patient.patientonsetageunit,patient.ageGroupCalculated,patient.ageGroupCalculated_was_imputed
0,24916426,1,NZ,ESTRADIOL,1,ESTRADIOL,1,female,52.0,801.0,adult,0
1,24917560,1,CA,CARBAMAZEPINE | CENOBAMATE | LAMOTRIGINE | OXC...,1 | 2,CARBAMAZEPINE | LAMOTRIGINE | OXCARBAZEPINE | ...,12,unknown,NaN,NaN,adult,1
2,24917803,2,US,DUPILUMAB,1,DUPIXENT,2,male,63.0,801.0,elderly,0
3,24920814,1,FR,ACETAMINOPHEN | AMPHOTERICIN B | HEPARIN CALCI...,1 | 2,ACETAMINOPHEN | AMPHOTERICIN B | HEPARIN CALCI...,7,male,53.0,801.0,adult,0
4,24921071,1,AU,ACETAMINOPHEN | CLONIDINE HYDROCHLORIDE | DULO...,1,ACETAMINOPHEN | CLONIDINE HYDROCHLORIDE | DULO...,8,male,52.0,801.0,adult,0


## 2. Recriacao da Faixa Etaria Antes da Imputacao

A coluna `patient.ageGroupCalculated` do CSV anterior nao é reutilizada, porque ela ja estava imputada pelo KNN com o dataset inteiro. Aqui a faixa etaria é recalculada apenas a partir da idade/unidade originais; registros sem idade valida ficam como `unknown` ate a etapa de imputacao apos o split.


In [11]:
def age_to_years(age, unit):
    if pd.isna(age) or pd.isna(unit):
        return np.nan

    try:
        unit_code = int(float(unit))
    except (TypeError, ValueError):
        return np.nan

    if unit_code == 800:      # decada
        return age * 10
    if unit_code == 801:      # ano
        return age
    if unit_code == 802:      # mes
        return age / 12
    if unit_code == 803:      # semana
        return age / 52
    if unit_code == 804:      # dia
        return age / 365
    if unit_code == 805:      # hora
        return age / (365 * 24)

    return np.nan


def calculate_age_group(age_years):
    if pd.isna(age_years):
        return "unknown"
    if 0 <= age_years < 2:
        return "baby_early_childhood"
    if 2 <= age_years < 12:
        return "child"
    if 12 <= age_years < 18:
        return "adolescent"
    if 18 <= age_years < 30:
        return "young_adult"
    if 30 <= age_years < 60:
        return "adult"
    if age_years >= 60:
        return "elderly"
    return "unknown"

base_df["patient.patientonsetage"] = pd.to_numeric(
    base_df["patient.patientonsetage"], errors="coerce"
)

age_years = base_df.apply(
    lambda row: age_to_years(
        row["patient.patientonsetage"],
        row["patient.patientonsetageunit"],
    ),
    axis=1,
)
age_years = age_years.mask((age_years < 0) | (age_years > 120))

prepared_df = base_df.copy()
prepared_df["patient.ageGroupCalculated"] = age_years.apply(calculate_age_group)
prepared_df["patient.ageGroupCalculated_was_imputed"] = (
    prepared_df["patient.ageGroupCalculated"] == "unknown"
).astype(int)

print("Distribuicao da faixa etaria antes do KNN:")
print(prepared_df["patient.ageGroupCalculated"].value_counts(dropna=False))
print("\nRegistros que precisarao de imputacao:")
print(prepared_df["patient.ageGroupCalculated_was_imputed"].value_counts(dropna=False))


Distribuicao da faixa etaria antes do KNN:
patient.ageGroupCalculated
unknown                 12726
elderly                  9472
adult                    7271
young_adult              1452
adolescent                617
child                     605
baby_early_childhood      166
Name: count, dtype: int64

Registros que precisarao de imputacao:
patient.ageGroupCalculated_was_imputed
0    19583
1    12726
Name: count, dtype: int64


## 3. Analise Descritiva dos Atributos

Esta analise resume a base antes da divisao e antes da imputacao KNN revisada.


In [12]:
print("=" * 80)
print("ANALISE DESCRITIVA DOS ATRIBUTOS")
print("=" * 80)

print("\nDimensoes do dataset base:")
print(f"  Total de registros: {prepared_df.shape[0]}")
print(f"  Total de atributos: {prepared_df.shape[1]}")

print("\nTipos de dados:")
print(prepared_df.dtypes)

print("\nValores unicos por atributo:")
for col in prepared_df.columns:
    print(f"  {col}: {prepared_df[col].nunique(dropna=False)}")

print("\nDistribuicao da variavel alvo 'serious':")
serious_counts = prepared_df["serious"].value_counts(dropna=False).sort_index()
serious_pct = prepared_df["serious"].value_counts(dropna=False, normalize=True).sort_index() * 100
for val, count in serious_counts.items():
    print(f"  {val}: {count} ({serious_pct[val]:.2f}%)")

print("\nEstatisticas de 'patient.drug.count':")
print(prepared_df["patient.drug.count"].describe())

print("\nTop 10 paises de ocorrencia:")
print(prepared_df["occurcountry"].value_counts().head(10))

print("\nDistribuicao de sexo:")
print(prepared_df["patient.patientsex"].value_counts(dropna=False))


ANALISE DESCRITIVA DOS ATRIBUTOS

Dimensoes do dataset base:
  Total de registros: 32309
  Total de atributos: 12

Tipos de dados:
safetyreportid                                        int64
serious                                               int32
occurcountry                                         object
patient.drug.activesubstance.activesubstancename     object
patient.drug.drugcharacterization                    object
patient.drug.medicinalproduct                        object
patient.drug.count                                    int64
patient.patientsex                                   object
patient.patientonsetage                             float64
patient.patientonsetageunit                         float64
patient.ageGroupCalculated                           object
patient.ageGroupCalculated_was_imputed                int32
dtype: object

Valores unicos por atributo:
  safetyreportid: 32309
  serious: 2
  occurcountry: 112
  patient.drug.activesubstance.activesubstancena

## 4. Divisao em Treinamento e Teste

A divisao 70/30 é feita antes do KNN, feature engineering e normalizacao. A estratificaeeo preserva a proporcao da variavel alvo `serious` nos dois conjuntos.


In [13]:
df_train, df_test = train_test_split(
    prepared_df,
    test_size=0.30,
    random_state=42,
    stratify=prepared_df["serious"],
)

print(f"Conjunto de treinamento: {df_train.shape[0]} registros ({df_train.shape[0] / len(prepared_df) * 100:.1f}%)")
print(f"Conjunto de teste: {df_test.shape[0]} registros ({df_test.shape[0] / len(prepared_df) * 100:.1f}%)")

print("\nDistribuicao de 'serious' no treino:")
print(df_train["serious"].value_counts(dropna=False).sort_index())
print("\nDistribuicao de 'serious' no teste:")
print(df_test["serious"].value_counts(dropna=False).sort_index())


Conjunto de treinamento: 22616 registros (70.0%)
Conjunto de teste: 9693 registros (30.0%)

Distribuicao de 'serious' no treino:
serious
1    14089
2     8527
Name: count, dtype: int64

Distribuicao de 'serious' no teste:
serious
1    6039
2    3654
Name: count, dtype: int64


## 5. Imputacao KNN Sem Vazamento de Dados

O KNN aprende somente com os registros do treino que possuem faixa etaria conhecida. Em seguida, o mesmo pipeline é usado para preencher os registros `unknown` do treino e do teste. A variavel alvo `serious` nao entra como atributo do KNN.


In [14]:
knn_features = [
    "occurcountry",
    "patient.patientsex",
    "patient.drug.drugcharacterization",
    "patient.drug.activesubstance.activesubstancename",
    "patient.drug.medicinalproduct",
    "patient.drug.count",
]

categorical_features = [
    "occurcountry",
    "patient.patientsex",
    "patient.drug.drugcharacterization",
]
text_active_substance_feature = "patient.drug.activesubstance.activesubstancename"
text_medicinal_product_feature = "patient.drug.medicinalproduct"
knn_numeric_features = ["patient.drug.count"]

train_known_mask = df_train["patient.ageGroupCalculated"] != "unknown"
if train_known_mask.sum() == 0:
    raise ValueError("Neo he registros conhecidos no treino para treinar o KNN.")

knn_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        (
            "active_substance",
            CountVectorizer(max_features=100, token_pattern=r"\b[\w-]+\b"),
            text_active_substance_feature,
        ),
        (
            "medicinal_product",
            CountVectorizer(max_features=100, token_pattern=r"\b[\w-]+\b"),
            text_medicinal_product_feature,
        ),
        ("num", StandardScaler(), knn_numeric_features),
    ]
)

knn_imputer = Pipeline(
    steps=[
        ("preprocessor", knn_preprocessor),
        (
            "knn",
            KNeighborsClassifier(
                n_neighbors=5,
                weights="distance",
                algorithm="brute",
                metric="euclidean",
            ),
        ),
    ]
)

knn_imputer.fit(
    df_train.loc[train_known_mask, knn_features],
    df_train.loc[train_known_mask, "patient.ageGroupCalculated"],
)

set_config(working_memory=32)


def apply_age_group_imputation(df, fitted_imputer, batch_size=500):
    df_imputed = df.copy()
    unknown_mask = df_imputed["patient.ageGroupCalculated"] == "unknown"
    unknown_indices = df_imputed.index[unknown_mask]

    predictions = []
    for start in range(0, len(unknown_indices), batch_size):
        batch_indices = unknown_indices[start:start + batch_size]
        batch_predictions = fitted_imputer.predict(df_imputed.loc[batch_indices, knn_features])
        predictions.extend(batch_predictions)

    if len(unknown_indices) > 0:
        df_imputed.loc[unknown_indices, "patient.ageGroupCalculated"] = predictions

    return df_imputed


df_train_imputed = apply_age_group_imputation(df_train, knn_imputer)
df_test_imputed = apply_age_group_imputation(df_test, knn_imputer)

print("Registros imputados por conjunto:")
print(f"  Treino: {df_train_imputed['patient.ageGroupCalculated_was_imputed'].sum()}")
print(f"  Teste: {df_test_imputed['patient.ageGroupCalculated_was_imputed'].sum()}")
print("\nDistribuicao depois da imputacao - treino:")
print(df_train_imputed["patient.ageGroupCalculated"].value_counts(dropna=False))
print("\nDistribuicao depois da imputacao - teste:")
print(df_test_imputed["patient.ageGroupCalculated"].value_counts(dropna=False))


Registros imputados por conjunto:
  Treino: 8945
  Teste: 3781

Distribuicao depois da imputacao - treino:
patient.ageGroupCalculated
elderly                 11585
adult                    8544
young_adult              1257
adolescent                554
child                     520
baby_early_childhood      156
Name: count, dtype: int64

Distribuicao depois da imputacao - teste:
patient.ageGroupCalculated
elderly                 5027
adult                   3615
young_adult              550
child                    226
adolescent               216
baby_early_childhood      59
Name: count, dtype: int64


## 6. Feature Engineering

As novas caracteristicas sao deterministicas e derivadas de atributos ja disponiveis em cada registro.


In [15]:
def count_pipe_values(value):
    if pd.isna(value) or str(value).strip().lower() == "unknown":
        return 0
    return len([part for part in str(value).split("|") if part.strip()])


def create_features(df):
    df_features = df.copy()

    df_features["feature_n_active_substances"] = (
        df_features["patient.drug.activesubstance.activesubstancename"].apply(count_pipe_values)
    )
    df_features["feature_n_drug_types"] = (
        df_features["patient.drug.drugcharacterization"].apply(count_pipe_values)
    )
    df_features["feature_has_medicinal_product"] = (
        df_features["patient.drug.medicinalproduct"].str.lower().ne("unknown").astype(int)
    )
    df_features["feature_multiple_drugs"] = (
        df_features["patient.drug.count"] > 1
    ).astype(int)
    df_features["feature_is_usa"] = (
        df_features["occurcountry"].str.upper().eq("US")
    ).astype(int)

    return df_features


df_train_features = create_features(df_train_imputed)
df_test_features = create_features(df_test_imputed)

new_features = [col for col in df_train_features.columns if col.startswith("feature_")]
print("Novas caracteristicas criadas:")
for feature in new_features:
    print(f"\n  {feature}:")
    print(f"    Treino - min: {df_train_features[feature].min():.0f}, max: {df_train_features[feature].max():.0f}, media: {df_train_features[feature].mean():.2f}")
    print(f"    Teste  - min: {df_test_features[feature].min():.0f}, max: {df_test_features[feature].max():.0f}, media: {df_test_features[feature].mean():.2f}")


Novas caracteristicas criadas:

  feature_n_active_substances:
    Treino - min: 1, max: 95, media: 3.29
    Teste  - min: 1, max: 115, media: 3.28

  feature_n_drug_types:
    Treino - min: 1, max: 3, media: 1.34
    Teste  - min: 1, max: 3, media: 1.34

  feature_has_medicinal_product:
    Treino - min: 1, max: 1, media: 1.00
    Teste  - min: 1, max: 1, media: 1.00

  feature_multiple_drugs:
    Treino - min: 0, max: 1, media: 0.64
    Teste  - min: 0, max: 1, media: 0.64

  feature_is_usa:
    Treino - min: 0, max: 1, media: 0.60
    Teste  - min: 0, max: 1, media: 0.60


## 7. Normalizacao

O `StandardScaler` é ajustado exclusivamente no conjunto de treinamento. As medias e desvios aprendidos sao salvos nos metadados e aplicados ao teste.


In [16]:
numeric_features = [
    "patient.drug.count",
    "feature_n_active_substances",
    "feature_n_drug_types",
]

scaler = StandardScaler()
scaler.fit(df_train_features[numeric_features])

df_train_transformed = df_train_features.copy()
df_test_transformed = df_test_features.copy()

df_train_transformed[numeric_features] = scaler.transform(df_train_features[numeric_features])
df_test_transformed[numeric_features] = scaler.transform(df_test_features[numeric_features])

scaler_params = {
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist(),
    "features": numeric_features,
}

print("Parametros aprendidos no TREINO:")
print(f"  Media: {scaler.mean_}")
print(f"  Desvio padrao: {scaler.scale_}")

print("\nEstatisticas apos normalizacao - TREINO:")
print(df_train_transformed[numeric_features].describe())
print("\nEstatisticas apos normalizacao - TESTE:")
print(df_test_transformed[numeric_features].describe())


Parametros aprendidos no TREINO:
  Media: [4.53762823 3.29288999 1.34475593]
  Desvio padrao: [11.07392271  4.49684428  0.47788639]

Estatisticas apos normalizacao - TREINO:
       patient.drug.count  feature_n_active_substances  feature_n_drug_types
count        2.261600e+04                 2.261600e+04          2.261600e+04
mean        -2.010733e-17                 6.472046e-17          1.558318e-16
std          1.000022e+00                 1.000022e+00          1.000022e+00
min         -3.194557e-01                -5.098887e-01         -7.214182e-01
25%         -3.194557e-01                -5.098887e-01         -7.214182e-01
50%         -2.291535e-01                -2.875105e-01         -7.214182e-01
75%          4.175321e-02                 1.572458e-01          1.371129e+00
max          7.436049e+01                 2.039366e+01          3.463677e+00

Estatisticas apos normalizacao - TESTE:
       patient.drug.count  feature_n_active_substances  feature_n_drug_types
count         9

## 8. Exportacao dos Datasets e Metadados


In [17]:
output_dir.mkdir(parents=True, exist_ok=True)

train_path = output_dir / "drug_events_train_transformed.csv"
test_path = output_dir / "drug_events_test_transformed.csv"
sample_path = output_dir / "drug_events_sample.csv"


# Mantem a mesma estrutura em treino e teste.
df_train_transformed.to_csv(train_path, index=False)
df_test_transformed.to_csv(test_path, index=False)

sample_train = df_train_transformed.head(25).copy()
sample_train.insert(0, "split", "train")
sample_test = df_test_transformed.head(25).copy()
sample_test.insert(0, "split", "test")
sample_df = pd.concat([sample_train, sample_test], ignore_index=True)
sample_df.to_csv(sample_path, index=False)

# Salva metadados: parametros aprendidos no treino
metadata = {
    "scaler_parameters": scaler_params,
    "age_group_imputation": {
        "method": "KNNImputer",
        "fit_scope": "training set only",
        "target": "patient.ageGroupCalculated",
        "train_imputed_records": int(df_train_features["patient.ageGroupCalculated_was_imputed"].sum()),
        "test_imputed_records": int(df_test_features["patient.ageGroupCalculated_was_imputed"].sum()),
    },
    "dataset_info": {
        "numeric_features_normalized": numeric_features,
        "train_shape": df_train_transformed.shape,
        "test_shape": df_test_transformed.shape,
    }
}

metadata_path = output_dir / "data_preparation_metadata.json"
with metadata_path.open('w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)


print("Arquivos exportados:")
print(f"  {train_path} ({len(df_train_transformed)} registros)")
print(f"  {test_path} ({len(df_test_transformed)} registros)")
print(f"  {sample_path} ({len(sample_df)} registros)")
print(f"  {metadata_path.name} (parametros aprendidos no treino)")



Arquivos exportados:
  c:\Users\Joao Vitor\Desktop\mineracao-dados\datamining-project-unifei\outputs\drug_events_train_transformed.csv (22616 registros)
  c:\Users\Joao Vitor\Desktop\mineracao-dados\datamining-project-unifei\outputs\drug_events_test_transformed.csv (9693 registros)
  c:\Users\Joao Vitor\Desktop\mineracao-dados\datamining-project-unifei\outputs\drug_events_sample.csv (50 registros)
  data_preparation_metadata.json (parametros aprendidos no treino)


## 9. Amostra do Dataset


In [18]:
print("Amostra exportada para conferencia:")
preview_columns = [
    "split",
    "safetyreportid",
    "serious",
    "occurcountry",
    "patient.patientsex",
    "patient.ageGroupCalculated",
    "patient.ageGroupCalculated_was_imputed",
    "patient.drug.count",
    "feature_n_active_substances",
    "feature_n_drug_types",
]
print(sample_df[preview_columns].head(10))

print("\nEntrega 2 revisada concluida.")


Amostra exportada para conferencia:
   split  safetyreportid  serious occurcountry patient.patientsex  \
0  train        23530731        1           IN             female   
1  train        23573794        1           US               male   
2  train        24801725        1           US             female   
3  train        24830941        1           US               male   
4  train        21927215        1           US               male   
5  train        21883320        2           US             female   
6  train        23678680        2           US            unknown   
7  train        22084593        1           FR             female   
8  train        21800099        2           US             female   
9  train        23409935        1           AU               male   

  patient.ageGroupCalculated  patient.ageGroupCalculated_was_imputed  \
0                young_adult                                       1   
1                    elderly                                